# Advanced Reinforcement Learning

## Model-Based RL

Instead of learning purely from interactions, **learn a model** $\hat{P}(s'|s,a)$ and use it for planning.

### Dyna-Q
Combines model-free Q-learning with model-based planning:
1. Take action, observe $(s,a,r,s')$
2. Update Q directly (model-free)
3. Update model $\hat{P}$
4. Do $n$ planning steps using model: sample $(s,a)$, predict $r,s'$, update Q

### World Models
Learn a latent space model:
- **Vision model** (V): $z_t = \text{enc}(x_t)$ compress observations to latent
- **Memory model** (M): $h_t = \text{RNN}(h_{t-1}, z_t, a_t)$ predict future latents
- **Controller** (C): $a_t = \pi(h_t, z_t)$ linear policy in latent space

### MuZero
Learns everything without a known environment model:
- Representation function: $s_0 = h(o)$
- Dynamics function: $r_t, s_t = g(s_{t-1}, a_t)$
- Prediction function: $p_t, v_t = f(s_t)$
Uses MCTS for planning with learned model.

---

## Hierarchical RL

Decompose long-horizon tasks into sub-tasks using **options** (temporally extended actions).

**Options framework**: An option $o = (I_o, \pi_o, \beta_o)$ where:
- $I_o \subseteq S$: initiation set (when can the option start)
- $\pi_o$: option policy (intra-option policy)
- $\beta_o(s)$: termination condition

**HIRO** (Hierarchical RL with Off-policy correction):
- High-level policy sets goals for low-level policy
- Low-level policy executes actions to reach goals

---

## Multi-Agent RL (MARL)

Multiple agents interact in shared environment. Challenges:
- Non-stationarity (other agents change)
- Cooperation vs competition
- Credit assignment

### MADDPG (Multi-Agent DDPG)
**Centralized training, decentralized execution**:
- During training: critic uses all agents' observations and actions
- During execution: each agent acts based only on local observation

$$Q_i(s_1,...,s_n, a_1,...,a_n)$$

### QMIX
For cooperative settings factorize joint Q-value:

$$Q_{tot}(\tau, a) = f_{mix}(Q_1(\tau_1,a_1),...,Q_n(\tau_n,a_n))$$

where $f_{mix}$ is a monotonic mixing network (weights are non-negative).

---

## Imitation Learning & Inverse RL

Learn from demonstrations rather than reward signals.

### Behavioral Cloning (BC)
Supervised learning on expert demonstrations:

$$\min_\theta \mathbb{E}_{(s,a)\sim \mathcal{D}_{expert}}[-\log \pi_\theta(a|s)]$$

Problem: distribution shift (covariate shift).

### DAgger (Dataset Aggregation)
Iteratively collect data under current policy, label with expert, retrain.

### GAIL (Generative Adversarial Imitation Learning)
Uses GAN framework discriminator distinguishes agent from expert:

$$\min_\pi \max_D \mathbb{E}_\pi[\log D(s,a)] + \mathbb{E}_{\pi_E}[\log(1-D(s,a))] - \lambda H(\pi)$$

### Inverse RL
Recover the reward function from expert demonstrations, then solve for optimal policy.

---

## Offline RL

Learn from a **fixed dataset** without environment interaction. Challenges: distributional shift, OOD actions.

### CQL (Conservative Q-Learning)
Penalize Q-values for out-of-distribution actions:

$$\min_Q \alpha \mathbb{E}_{s\sim\mathcal{D}}[\log\sum_a e^{Q(s,a)}] - \mathbb{E}_{(s,a)\sim\mathcal{D}}[Q(s,a)] + \frac{1}{2}L_{TD}(Q)$$

### Decision Transformer
Frames RL as **sequence modeling**: given target return, past states and actions, predict next action:

$$\hat{a}_t = \text{Transformer}(\hat{R}_t, s_t, a_{t-1}, \hat{R}_{t-1}, s_{t-1}, ...)$$

---

## Meta-RL

Learn to learn adapt quickly to new tasks with few samples.

### MAML (Model-Agnostic Meta-Learning)

$$\theta^* = \theta - \alpha \nabla_\theta L_{\tau_i}(f_\theta)$$

$$\theta \leftarrow \theta - \beta \nabla_\theta \sum_{\tau_i} L_{\tau_i}(f_{\theta^*_i})$$

Find initial parameters that can be quickly adapted to any task in the distribution.

---

## Curiosity-Driven Exploration

**ICM (Intrinsic Curiosity Module)**:
- **Feature encoder**: $\phi(s_t)$
- **Forward model**: $\hat{\phi}(s_{t+1}) = f(\phi(s_t), a_t)$
- **Inverse model**: $\hat{a}_t = g(\phi(s_t), \phi(s_{t+1}))$
- **Intrinsic reward**: $r_t^i = \frac{\eta}{2}\|\hat{\phi}(s_{t+1}) - \phi(s_{t+1})\|_2^2$

High prediction error → unexplored → high intrinsic reward.

---

## RLHF (Reinforcement Learning from Human Feedback)

Used in ChatGPT, Claude, Gemini. Three stages:

1. **SFT** (Supervised Fine-Tuning): fine-tune LLM on demonstration data
2. **Reward Model**: train $R_\phi(x,y)$ on human preference pairs $(y_w \succ y_l)$:
   $$L = -\mathbb{E}\left[\log \sigma(R_\phi(x,y_w) - R_\phi(x,y_l))\right]$$
3. **RL Fine-Tuning** (PPO): optimize LLM against reward model with KL penalty:
   $$R(x,y) = R_\phi(x,y) - \beta D_{KL}(\pi_\theta(y|x) \| \pi_{ref}(y|x))$$

In [1]:
# ─── PPO with Stable-Baselines3 on more complex envs ───
try:
    from stable_baselines3 import PPO, SAC
    from stable_baselines3.common.callbacks import EvalCallback
    from stable_baselines3.common.env_util import make_vec_env
    import gymnasium as gym

    # LunarLander with PPO
    print("Training PPO on LunarLander-v3...")
    env = make_vec_env('LunarLander-v3', n_envs=4)
    model = PPO('MlpPolicy', env, verbose=0, learning_rate=3e-4,
                n_steps=1024, batch_size=64, n_epochs=4, gamma=0.999,
                gae_lambda=0.98, clip_range=0.2, ent_coef=0.01)
    model.learn(total_timesteps=200_000)
    print("PPO trained on LunarLander-v3")

    # Evaluate
    test_env = gym.make('LunarLander-v3')
    rewards = []
    for _ in range(10):
        obs, _ = test_env.reset()
        total = 0
        for _ in range(1000):
            action, _ = model.predict(obs, deterministic=True)
            obs, r, t, tr, _ = test_env.step(action)
            total += r
            if t or tr: break
        rewards.append(total)
    print(f"Eval avg reward: {sum(rewards)/len(rewards):.1f} (solved=200)")
    test_env.close()

except ImportError:
    print("Install: pip install stable-baselines3 gymnasium[box2d]")

Training PPO on LunarLander-v3...


PPO trained on LunarLander-v3


Eval avg reward: -24.7 (solved=200)


In [2]:
# ─── Behavioral Cloning Example ───
import torch
import torch.nn as nn
import numpy as np

# Generate expert demonstrations using pre-trained PPO (or random for demo)
print("Behavioral Cloning concept:")
print("1. Collect demonstrations: (state, expert_action) pairs")
print("2. Train policy network with cross-entropy loss")
print("3. Issue: distribution shift during rollout")
print("4. Fix: DAgger iteratively relabel with expert")

# Simple BC implementation
class BCPolicy(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
            nn.Linear(hidden, act_dim)
        )
    def forward(self, x): return self.net(x)

# Synthetic demo data
obs_dim, act_dim = 4, 2
demo_obs  = torch.randn(1000, obs_dim)
demo_acts = torch.randint(0, act_dim, (1000,))

bc_policy = BCPolicy(obs_dim, act_dim)
optimizer = torch.optim.Adam(bc_policy.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(100):
    logits = bc_policy(demo_obs)
    loss = criterion(logits, demo_acts)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

print(f"\nBC final loss: {loss.item():.4f}")
print("Accuracy:", (bc_policy(demo_obs).argmax(1) == demo_acts).float().mean().item())

Behavioral Cloning concept:
1. Collect demonstrations: (state, expert_action) pairs
2. Train policy network with cross-entropy loss
3. Issue: distribution shift during rollout
4. Fix: DAgger iteratively relabel with expert



BC final loss: 0.6774
Accuracy: 0.5709999799728394


## Additional Learning Resources

### Papers
- **World Models**: https://arxiv.org/abs/1803.10122
- **MuZero**: https://arxiv.org/abs/1911.08265
- **MADDPG**: https://arxiv.org/abs/1706.02275
- **GAIL**: https://arxiv.org/abs/1606.03476
- **CQL** (Offline RL): https://arxiv.org/abs/2006.04779
- **Decision Transformer**: https://arxiv.org/abs/2106.01345
- **MAML**: https://arxiv.org/abs/1703.03400
- **ICM** (Curiosity): https://arxiv.org/abs/1705.05363
- **InstructGPT (RLHF)**: https://arxiv.org/abs/2203.02155

### Courses & Blogs
- **Lilian Weng Policy Gradient Algorithms**: https://lilianweng.github.io/posts/2018-04-08-policy-gradient/
- **Lilian Weng Exploration Strategies**: https://lilianweng.github.io/posts/2020-06-07-exploration-drl/
- **Spinning Up** Complete RL algorithms: https://spinningup.openai.com/

### Libraries
- **Stable-Baselines3**: https://stable-baselines3.readthedocs.io/
- **RLlib** (scalable, distributed): https://docs.ray.io/en/latest/rllib/
- **TorchRL**: https://github.com/pytorch/rl
- **d3rlpy** (offline RL): https://github.com/takuseno/d3rlpy